In [58]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [59]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Drop columns
drop_cols = ['id', 'date', 'zipcode', 'Unnamed: 0']
train = train.drop(columns=[c for c in drop_cols if c in train.columns])
test = test.drop(columns=[c for c in drop_cols if c in test.columns])

# Separate features and target variable
X_train = train.drop(columns=['price'])
y_train = train['price'] / 1000

X_test = test.drop(columns=['price'])
y_test = test['price'] / 1000

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame
feature_names = X_train.columns
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_names)

# Problem 2

In [60]:
# Part 1: 
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Coefficients
print("=== Model Coefficients ===")
print(f"Intercept (theta_0): {model.intercept_:.4f}")
print("\nFeature Coefficients:")
for name, coef in zip(feature_names, model.coef_):
    print(f"  {name:20s}: {coef:.4f}")

# Training metrics
y_train_pred = model.predict(X_train_scaled)
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

print(f"\n=== Training Metrics ===")
print(f"MSE:  {train_mse:.4f}")
print(f"R²:   {train_r2:.4f}")

=== Model Coefficients ===
Intercept (theta_0): 520.4148

Feature Coefficients:
  bedrooms            : -12.5220
  bathrooms           : 18.5276
  sqft_living         : 56.7488
  sqft_lot            : 10.8819
  floors              : 8.0437
  waterfront          : 63.7429
  view                : 48.2001
  condition           : 12.9643
  grade               : 92.2315
  sqft_above          : 48.2901
  sqft_basement       : 27.1370
  yr_built            : -67.6431
  yr_renovated        : 17.2714
  lat                 : 78.3757
  long                : -1.0352
  sqft_living15       : 45.5777
  sqft_lot15          : -12.9301

=== Training Metrics ===
MSE:  31486.1678
R²:   0.7265


In [61]:
# Part 2:
y_test_pred = model.predict(X_test_scaled)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"\n=== Testing Metrics ===")
print(f"MSE:  {test_mse:.4f}")
print(f"R²:   {test_r2:.4f}")


=== Testing Metrics ===
MSE:  57628.1547
R²:   0.6544


### Part 3: Interpretation

The features that contribute most to the linear regression model are grade (92.23), lat (78.38), yr_built (−67.64), waterfront (63.74), and sqft_living (56.75). The model fits the training data reasonably well with an R² of 0.7265, meaning it explains about 73% of the variance in house prices. However, the training MSE of 31,486 compared to the testing MSE of 57,628 (roughly 1.83× larger) indicates some degree of overfitting. This is reflected in the drop in R² from 0.73 (train) to 0.65 (test). The model generalizes to unseen data but with reduced accuracy

# Problem 3

In [62]:
# Part 1: Closed Form Solution
def fit_closed_form(X, y):
    """ Train linear regression using the closed form solution """
    X_b = np.column_stack([np.ones(X.shape[0]), X])
    theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
    return theta

def predict(X, theta):
    """ Predict using theta """
    X_b = np.column_stack([np.ones(X.shape[0]), X])
    return X_b @ theta

# Train and Evaluate
theta = fit_closed_form(np.array(X_train_scaled), y_train.values)

# Report coefficients
print("=== Closed-Form Coefficients ===")
print(f"Intercept (theta_0): {theta[0]:.4f}")
print("\nFeature Coefficients:")
for name, coef in zip(feature_names, theta[1:]):
    print(f"  {name:20s}: {coef:.4f}")

# Training metrics
y_train_pred_cf = predict(np.array(X_train_scaled), theta)
cf_train_mse = mean_squared_error(y_train, y_train_pred_cf)
cf_train_r2 = r2_score(y_train, y_train_pred_cf)

print(f"\n=== Training Metrics ===")
print(f"MSE:  {cf_train_mse:.4f}")
print(f"R²:   {cf_train_r2:.4f}")

# Testing metrics
y_test_pred_cf = predict(np.array(X_test_scaled), theta)
cf_test_mse = mean_squared_error(y_test, y_test_pred_cf)
cf_test_r2 = r2_score(y_test, y_test_pred_cf)

print(f"\n=== Testing Metrics ===")
print(f"MSE:  {cf_test_mse:.4f}")
print(f"R²:   {cf_test_r2:.4f}")

=== Closed-Form Coefficients ===
Intercept (theta_0): 520.4148

Feature Coefficients:
  bedrooms            : -12.5220
  bathrooms           : 18.5276
  sqft_living         : 56.7488
  sqft_lot            : 10.8819
  floors              : 8.0437
  waterfront          : 63.7429
  view                : 48.2001
  condition           : 12.9643
  grade               : 92.2315
  sqft_above          : 48.2901
  sqft_basement       : 27.1370
  yr_built            : -67.6431
  yr_renovated        : 17.2714
  lat                 : 78.3757
  long                : -1.0352
  sqft_living15       : 45.5777
  sqft_lot15          : -12.9301

=== Training Metrics ===
MSE:  31486.1678
R²:   0.7265

=== Testing Metrics ===
MSE:  57628.1547
R²:   0.6544


### Part 2: Comparison

The closed-form implementation produces identical results to the sklearn's LinearRegression. Both models yield the same coefficients, with a training MSE of 31,486.17 (R² = 0.7265) and testing MSE of 57,628.15 (R² = 0.6544). This is expected, as sklearn's LinearRegression uses the same normal equation. 

# Problem 4

In [63]:
# Part 1:

# Extract sqft_living feature
X_train_sqft = np.array(X_train_scaled['sqft_living']).reshape(-1, 1)
X_test_sqft = np.array(X_test_scaled['sqft_living']).reshape(-1, 1)

# Polynomial Feature Generation
def make_poly_features(X, degree):
    """ Generate polynomial features [X, X^2, ..., X^degree] """
    return np.column_stack([X**d for d in range(1, degree + 1)])

# Train and evaluate for p = 1, 2, 3, 4, 5
results = []

for p in range(1, 6):
    # Generate polynomial features
    X_train_poly = make_poly_features(X_train_sqft, p)
    X_test_poly = make_poly_features(X_test_sqft, p)

    # Fit using closed form solution from Problem 3
    theta_poly = fit_closed_form(X_train_poly, y_train.values)

    # Predict
    y_train_pred_poly = predict(X_train_poly, theta_poly)
    y_test_pred_poly = predict(X_test_poly, theta_poly)

    # Metrics
    train_mse = mean_squared_error(y_train, y_train_pred_poly)
    train_r2 = r2_score(y_train, y_train_pred_poly)
    test_mse = mean_squared_error(y_test, y_test_pred_poly)
    test_r2 = r2_score(y_test, y_test_pred_poly)

    results.append({
        'Degree (p)' : p,
        'Train MSE'  : round(train_mse, 4),
        'Train R²'   : round(train_r2, 4),
        'Test MSE'   : round(test_mse, 4),
        'Test R²'    : round(test_r2, 4)
    })

# Display Results
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

 Degree (p)  Train MSE  Train R²    Test MSE  Test R²
          1 57947.5262    0.4967  88575.9785   0.4687
          2 54822.6651    0.5238  71791.6795   0.5694
          3 53785.1947    0.5329  99833.4838   0.4012
          4 52795.7748    0.5415 250979.2743  -0.5053
          5 52626.1120    0.5429 570616.9148  -2.4225


### Part 2: Observations

As the polynomial degree increases, training MSE decreases (from 57,948 at p=1 to 52,626 at p=5) and training R² increases (0.497 to 0.543). This is expected because higher degree polynomials have more flexibility to fit the training data. Test MSE initially improves from p=1 (88,576) to p=2 (71,792), which suggests that a quadratic relationship between sqft_living and price captures real curvature between sqft_living and price. But after p=2, performance gets much worse: test MSE jumps to 250,979 at p=4 and to 570,617 at p=5, and R² even becomes negative (-2.42). the higher-degree models are overfitting. They’re matching noise in the training data instead of the true pattern, so they don’t generalize well.

# Problem 5

In [64]:
# Part 1: Gradient Descent Implementation

def gradient_descent(X, y, alpha, n_iters):
    """ Gradient descent for linear regression
    Parameters:
        X : feature matrix (N x d)
        y: target vector (N,)
        alpha: learning rate
        n_iters: number of iterations
    Returns:
        theta: learned parameters
        history: dict with theta and metrics at iterations 10, 50, 100
    """
    # Add bias column
    X_b = np.column_stack([np.ones(X.shape[0]), X])
    N, d = X_b.shape

    # Initialize theta
    theta = np.zeros(d)

    # Track results at specific iterations
    checkpoints = [10, 50, 100]
    history = {}

    for i in range(1, n_iters + 1):
        # Predictions
        y_pred = X_b @ theta

        # Gradient: (2/N) * X^T * (y_pred - y)
        gradient = (2 / N) * (X_b.T @ (y_pred - y))

        # Update theta
        theta = theta - alpha * gradient

        # Save at checkpoints
        if i in checkpoints:
            history[i] = theta.copy()
    
    return theta, history


In [65]:
# Part 2: Vary Learning Rate and Iterations
X_train = np.array(X_train_scaled)
X_test = np.array(X_test_scaled)
y_train = y_train.values
y_test = y_test.values

learning_rates = [0.01, 0.1, 0.5]
checkpoints = [10, 50, 100]

results_gd = []

for alpha in learning_rates:
    # Run gradient descent for max iterations
    _, history = gradient_descent(X_train, y_train, alpha, max(checkpoints))

    for n_iter in checkpoints:
        theta_snap = history[n_iter]

        # Predict using snapshot theta
        y_train_pred = np.column_stack([np.ones(X_train.shape[0]), X_train]) @ theta_snap
        y_test_pred = np.column_stack([np.ones(X_test.shape[0]), X_test]) @ theta_snap

        train_mse = mean_squared_error(y_train, y_train_pred)
        train_r2 = r2_score(y_train, y_train_pred)
        test_mse = mean_squared_error(y_test, y_test_pred)
        test_r2 = r2_score(y_test, y_test_pred)

        results_gd.append({
            'Alpha': alpha,
            'Iterations': n_iter,
            'Train MSE': round(train_mse, 4),
            'Train R²': round(train_r2, 4),
            'Test MSE': round(test_mse, 4),
            'Test R²': round(test_r2, 4)
        })

# Display Results
results_gd_df = pd.DataFrame(results_gd)
print(results_gd_df.to_string(index=False))

 Alpha  Iterations     Train MSE       Train R²      Test MSE        Test R²
  0.01          10  2.357278e+05  -1.047400e+00  2.805687e+05  -6.828000e-01
  0.01          50  6.972050e+04   3.945000e-01  9.704954e+04   4.179000e-01
  0.01         100  3.682035e+04   6.802000e-01  6.333304e+04   6.201000e-01
  0.10          10  3.510510e+04   6.951000e-01  6.163043e+04   6.304000e-01
  0.10          50  3.149726e+04   7.264000e-01  5.772248e+04   6.538000e-01
  0.10         100  3.148643e+04   7.265000e-01  5.763896e+04   6.543000e-01
  0.50          10  1.456064e+17  -1.264635e+12  1.626068e+17  -9.752880e+11
  0.50          50  1.259542e+67  -1.093949e+62  1.406601e+67  -8.436553e+61
  0.50         100 3.322792e+129 -2.885942e+124 3.710745e+129 -2.225642e+124


### Part 3: Observations

Learning rate α = 0.01: The model converges slowly. At 10 iterations, the model is still poor (negative R²), but it gradually improves. By 100 iterations it reaches a Train R² of 0.68 and Test R² of 0.62. However, it still hasn't converged to the optimal solution (compared to the closed-form Train MSE of 31,486 and R² of 0.7265), indicating that more iterations are needed with this small learning rate.

Learning rate α = 0.1: The model converges much faster. By just 10 iterations it already achieves Train R² = 0.695, and by 50 iterations it is very close to the closed-form optimum (Train MSE ≈ 31,497, R² ≈ 0.7264). Between 50 and 100 iterations there is almost no change, indicating the algorithm has effectively converged to the optimal solution.

Learning rate α = 0.5: The model diverges completely. MSE explodes to astronomically large values (10¹²⁹ by 100 iterations) and R² becomes extremely negative. This is because the step size is so large that the algorithm overshoots the minimum on each iteration, going further and further away rather than converging.

With an appropriate learning rate (α = 0.1), gradient descent converges to the same solution as the closed-form method. The key tradeoff is selecting a learning rate that is large enough for fast convergence but small enough to avoid divergence.

# Problem 6

In [66]:
# Part 2: Ridge Regression with Gradient Descent

def ridge_gradient_descent(X, y, alpha, lamda, n_iters):
    """ Gradient descent for ridge regression
    The gradient of J(θ) = (1/N)Σ(Xθ-y)² + λΣθⱼ² is:
        (2/N) * X^T(Xθ - y) + 2λθ
    """
    X_b = np.column_stack([np.ones(X.shape[0]), X])
    N, d = X_b.shape
    theta = np.zeros(d)

    for i in range(1, n_iters + 1):
        y_pred = X_b @ theta
        
        # Gradient of MSE part: (2/N) * X^T(Xθ - y)
        grad_mse = (2 / N) * (X_b.T @ (y_pred - y))
        
        # Gradient of regularization part: 2λθ (but not for bias)
        grad_reg = 2 * lamda * theta
        grad_reg[0] = 0  # Don't regularize bias term
        
        gradient = grad_mse + grad_reg
        theta = theta - alpha * gradient
    
    return theta

In [67]:
# Part 3: Linear vs Ridge Regression

np.random.seed(42)

# Simulate data
N = 1000
X_sim = np.random.uniform(-2, 2, size=(N, 1))
e_sim = np.random.normal(0, 2, size=N)
y_sim = 1 + 2 * X_sim.flatten() + e_sim

# True parameters: intercept = 1, slope = 2

# Standard Linear Regression
theta_lr = fit_closed_form(X_sim, y_sim)
y_pred_lr = predict(X_sim, theta_lr)
lr_mse = mean_squared_error(y_sim, y_pred_lr)
lr_r2 = r2_score(y_sim, y_pred_lr)

print("Standard Linear Regression")
print(f"Intercept: {theta_lr[0]:.4f}  Slope: {theta_lr[1]:.4f}")
print(f"MSE: {lr_mse:.4f}  R²: {lr_r2:.4f}")
print(f"(True values: Intercept=1, Slope=2)\n")

# Ridge Regression
lambdas = [1, 10, 100, 1000, 10000]

print("Ridge Regression")
print(f"{'Lambda':>10} {'Intercept':>12} {'Slope':>10} {'MSE':>12} {'R²':>10}")

results_ridge = []
for lam in lambdas:
    # Ridge closed-form: θ = (X^T X + λI)^{-1} X^T y
    X_b = np.column_stack([np.ones(N), X_sim])

    # Create λI but don't regularize bias
    reg_matrix = lam * np.eye(X_b.shape[1])
    reg_matrix[0, 0] = 0  # Don't regularize intercept

    theta_rdige = np.linalg.inv(X_b.T @ X_b + reg_matrix) @ X_b.T @ y_sim
    y_pred_ridge = X_b @ theta_rdige

    ridge_mse = mean_squared_error(y_sim, y_pred_ridge)
    ridge_r2 = r2_score(y_sim, y_pred_ridge)

    print(f"{lam:10d} {theta_rdige[0]:12.4f} {theta_rdige[1]:10.4f} {ridge_mse:12.4f} {ridge_r2:10.4f}")

    results_ridge.append({
        'Lambda': lam,
        'Intercept': round(theta_rdige[0], 4),
        'Slope': round(theta_rdige[1], 4),
        'MSE': round(ridge_mse, 4),
        'R²': round(ridge_r2, 4)
    })

results_ridge_df = pd.DataFrame(results_ridge)

Standard Linear Regression
Intercept: 1.1948  Slope: 1.9226
MSE: 3.8999  R²: 0.5639
(True values: Intercept=1, Slope=2)

Ridge Regression
    Lambda    Intercept      Slope          MSE         R²
         1       1.1947     1.9212       3.8999     0.5639
        10       1.1942     1.9086       3.9001     0.5639
       100       1.1897     1.7913       3.9234     0.5613
      1000       1.1631     1.1094       4.8021     0.4630
     10000       1.1288     0.2308       7.8044     0.1273


### Part 4: Observations

Standard linear regression gets parameters close to the true values (intercept = 1.1948 ≈ 1, slope = 1.9226 ≈ 2), with MSE of 3.90 and R² of 0.56. 

As the regularization parameter λ increases in ridge regression, the slope progresses towards zero from 1.92 (λ = 1) to 1.79 (λ=100) to 0.23 (λ=10000). The intercept remains relatively stable throughout. This shows that ridge regression penalizes large coefficients, pushing them towards zero. 

For small λ values (1 and 10), the results are nearly identical to standard linear regression, meaning the regularization has minimal effect. However, as λ grows larger (1000 and 10000), the model is forced to shrink the slope so aggressively that it moves far from the true value of 2, causing MSE to increase (from 3.90 to 7.80) and R² to drop (from 0.56 to 0.13). This shows underfitting as the regularization is too strong and prevents the model from capturing the true relationship in the data. 